In [3]:
import pandas as pd
import numpy as np

# 1. Ensure you are using the correct column name (plural 'Installments')
# 2. Remove '$', ',', and whitespace, then convert to float
# we use .astype(str) first just in case there are mixed types
df['Installments'] = df['Installments'].astype(str).str.replace('$', '', regex=False)
df['Installments'] = df['Installments'].str.replace(',', '', regex=False).str.strip()

# 3. Convert to numeric (errors='coerce' will turn any unfixable text into NaN)
df['Installments'] = pd.to_numeric(df['Installments'], errors='coerce')

# 4. Now perform the Feature Engineering
df['Monthly Income'] = np.exp(df['Log Annual Income']) / 12
df['Installment_as_Pct_Income'] = (df['Installments'] / df['Monthly Income']) * 100

print("Calculation Successful!")
df[['Installments', 'Monthly Income', 'Installment_as_Pct_Income']].head()

Calculation Successful!


,Installments,Monthly Income,Installment_as_Pct_Income
0,829.10,7083.333365,11.704941
1,228.22,5416.666673,4.213292
2,366.86,2666.666662,13.757250
3,162.34,7083.333365,2.291859
4,102.92,6733.333303,1.528515


In [4]:
# 1. Load your file
df = pd.read_csv('loan_data.csv')

# 2. Strip invisible spaces from column names to prevent KeyErrors
df.columns = df.columns.str.strip()

# 3. Define which column(s) should NOT be turned into numbers
# In this dataset, 'Purpose' is the only text-based column.
categorical_cols = ['Purpose']

# 4. Loop through every column in the dataframe
for col in df.columns:
    if col not in categorical_cols:
        # Step A: Force the column to string type so we can strip symbols
        # Step B: Remove '$' and ','
        # Step C: Convert to numeric (errors='coerce' turns bad data into NaN)
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip(),
            errors='coerce'
        )

# 5. Verify the results
print("Cleaning complete! All columns except 'Purpose' are now numeric.")
print(df.dtypes)
df.head()

Cleaning complete! All columns except 'Purpose' are now numeric.
Credit Policy                int64
Purpose                     object
Interest Rate              float64
Installments               float64
Log Annual Income          float64
Annual Income              float64
Debt to Income Ratio       float64
FICO Score                   int64
FICO Score Bucket          float64
Days with Credit Line        int64
Revolving Balance          float64
Revolving Utilization      float64
Inquiries Last 6 Months      int64
Delinquencies 2 Years        int64
Public Records               int64
Not Fully Paid               int64
Unnamed: 16                float64
Unnamed: 17                float64
Unnamed: 18                float64
Unnamed: 19                float64
dtype: object


,Credit Policy,Purpose,Interest Rate,Installments,Log Annual Income,Annual Income,Debt to Income Ratio,FICO Score,FICO Score Bucket,Days with Credit Line,Revolving Balance,Revolving Utilization,Inquiries Last 6 Months,Delinquencies 2 Years,Public Records,Not Fully Paid,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19
0,1,Debt Consolidation,NaN,829.10,11.350407,85000.0,NaN,737,NaN,5640,28854.0,NaN,0,0,0,0,NaN,NaN,NaN,NaN
1,1,Credit Card,NaN,228.22,11.082143,65000.0,NaN,707,NaN,2760,33623.0,NaN,0,0,0,0,NaN,NaN,NaN,NaN
2,1,Debt Consolidation,NaN,366.86,10.373491,32000.0,NaN,682,NaN,4710,3511.0,NaN,1,0,0,0,NaN,NaN,NaN,NaN
3,1,Debt Consolidation,NaN,162.34,11.350407,85000.0,NaN,712,NaN,2700,33667.0,NaN,1,0,0,0,NaN,NaN,NaN,NaN
4,1,Credit Card,NaN,102.92,11.299732,80800.0,NaN,667,NaN,4066,4740.0,NaN,0,1,0,0,NaN,NaN,NaN,NaN


In [9]:
# 1. Load your file
df = pd.read_csv('loan_data.csv')




import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split





# 2. Define the categorical columns to ignore
categorical_cols = ['Purpose']

# 2. KEY STEP: Convert Categorical Text to Numeric
# This turns 'Purpose' into multiple columns: 'Purpose_debt_consolidation', 'Purpose_credit_card', etc.
# Each column will have a 1 if true, 0 if false.
df = pd.get_dummies(df, columns=['Purpose'], drop_first=True)


# 3. Clean and Convert Loop
for col in df.columns:
    if col not in categorical_cols:
        # We strip $, %, and commas, then convert to numeric
        df[col] = pd.to_numeric(
            df[col].astype(str)
                   .str.replace('$', '', regex=False)
                   .str.replace('%', '', regex=False)
                   .str.replace(',', '', regex=False)
                   .str.strip(), 
            errors='coerce'
        )
df.columns = df.columns.str.strip()
df = df.drop(columns=['Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'FICO Score Bucket'])        

# 4. Optional: Standardize "Rates" to Decimals
# Currently, Interest Rate is 0.12 (decimal) but Utilization is 52.1 (whole number).
# To make them consistent (both decimals), run the following:
cols_to_divide = ['Interest Rate','Debt to Income Ratio', 'Revolving Utilization']
for col in cols_to_divide:
    # Only divide if the max value is > 1 (meaning it's not already a decimal)
    if df[col].max() > 1:
        df[col] = df[col] / 100

        

print("All columns cleaned successfully!")

# 1. Calculate Monthly Income from the Log Annual Income
import numpy as np
df['Monthly Income'] = np.exp(df['Log Annual Income']) / 12

# 2. Monthly Debt-to-Income Payment Ratio
# How much of their monthly take-home pay goes JUST to this loan?
df['Installment_as_Pct_Income'] = df['Installments'] / df['Monthly Income']




df.head()

All columns cleaned successfully!


,Credit Policy,Interest Rate,Installments,Log Annual Income,Annual Income,Debt to Income Ratio,FICO Score,Days with Credit Line,Revolving Balance,Revolving Utilization,...,Public Records,Not Fully Paid,Purpose_Credit Card,Purpose_Debt Consolidation,Purpose_Educational,Purpose_Home Improvement,Purpose_Major Purchase,Purpose_Small Business,Monthly Income,Installment_as_Pct_Income
0,1,0.1189,829.10,11.350407,85000.0,0.1948,737,5640,28854.0,0.521,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,7083.333365,0.117049
1,1,0.1071,228.22,11.082143,65000.0,0.1429,707,2760,33623.0,0.767,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,5416.666673,0.042133
2,1,0.1357,366.86,10.373491,32000.0,0.1163,682,4710,3511.0,0.256,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,2666.666662,0.137573
3,1,0.1008,162.34,11.350407,85000.0,0.0810,712,2700,33667.0,0.732,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,7083.333365,0.022919
4,1,0.1426,102.92,11.299732,80800.0,0.1497,667,4066,4740.0,0.395,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,6733.333303,0.015285


In [6]:
# 1. Define the model
model = LogisticRegression(class_weight='balanced', solver='liblinear')

# 2. Define Features (X) and Target (y)
X = df.drop('Not Fully Paid', axis=1)
y = df['Not Fully Paid']

# 3. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Fit (Train) the model
model.fit(X_train, y_train)

print("Model trained successfully!")

ValueError: could not convert string to float: '650-700'

In [ ]:
df.head()

In [ ]:
from sklearn.linear_model import LogisticRegression

# Setting class_weight='balanced' automatically adjusts for your 16% minority class
model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)

In [ ]:
import sklearn
print(f"sklearn is installed and located at: {sklearn.__file__}")
print(f"Version: {sklearn.__version__}")

In [10]:
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [11]:
!pip install seaborn

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 46.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 52.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [seaborn]m8/9 [seaborn]ib]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
